<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### CLIP classification experiments
OpenAI CLIP model is a classification model, which creates multimodal embedding space for images and text labels, allowing user to compare the similarity between them. This notebook is dedicated to different experiments that (maybe) will lead people to better understanding of that model.

#### Imports and constants

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import gc

In [ ]:
BATCH_SIZE = 128
IMAGENETTE_CLASSES = ['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']

#### Model initialization

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", dtype=torch.bfloat16, attn_implementation="sdpa").to('cuda')
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#### Imagenette dataset initialization

In [ ]:
transform = transforms.Compose([
    transforms.PILToTensor(),
])

imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True, transform=transform)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=True,
                                          num_workers=2,
                                          collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

#### Picking random sample and analyzing CLIP output

In [ ]:
image = data_loader.dataset[128][0]
sample_categories = ['A man with a fish', 'Respectable man', 'Epstein island', 'Well-grown fish']
inputs = processor(text=sample_categories, images=image, padding=True, return_tensors='pt').to('cuda')
outputs = model(**inputs)

plt.title(f'This is probably: {sample_categories[outputs.logits_per_image.argmax(dim=1)]}')
plt.imshow(image[0])


#### Retrieving embeddings manually
Each label we pass to CLIP is processes via CLIPTokenizer and padded to the longest label so we get a list of numbers which have low payload

In [ ]:
inputs['input_ids']

In [ ]:
model.get_text_features(inputs['input_ids']).pooler_output.detach()

#### Evaluating CLIP on Imagenette

In [ ]:
correct_classifications = 0
all_classifications = 0
predictions = torch.Tensor([], device='cpu')
truth = torch.Tensor([], device='cpu')

for batch, t in tqdm(data_loader):
  with torch.no_grad():
    inputs = processor(images=batch, text=IMAGENETTE_CLASSES, padding=True, return_tensors='pt').to('cuda')
    outputs = model(**inputs)
    predicted_cat = outputs.logits_per_image.argmax(dim=1).to('cpu')

    batch_truth = torch.Tensor(t).to('cpu')
    predictions = torch.cat((predictions, predicted_cat))
    truth = torch.cat((truth, batch_truth))

    correct_classifications += int((predicted_cat == batch_truth).sum())
    all_classifications += BATCH_SIZE


In [ ]:
freqs = Counter(truth.tolist())
plt.title('Imagenette class sizes')
plt.bar(freqs.keys(), freqs.values())

In [ ]:
print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')    # Actually I have no idea in which case we may need micro-averaging.
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')       # The classes are distributed rather equally, so we could use micro,
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')           # but macro seems to be at least not worse.